# 5.1 LoRA Fine-tuning - Customize the Agent's LLM

## Prerequisites: GPU 8GB+, model at ./models/Qwen3.7-1.7B-Instruct

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f}GB")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_PATH = "./models/Qwen3.7-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
print(f"Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")


In [ ]:
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
import json
TRAIN_DATA = [
    {"instruction":"What is RAG?","input":"","output":"RAG combines document retrieval with LLM generation for accurate sourced answers."},
    {"instruction":"What is MCP?","input":"","output":"MCP is AI Agent standard protocol by Linux Foundation. 3 primitives: Tool Resource Prompt."},
    {"instruction":"Explain LoRA","input":"","output":"LoRA trains low-rank matrices for efficient fine-tuning. Only ~0.1% params trained."},
]
with open("./data/train_data.json", "w") as f:
    json.dump(TRAIN_DATA, f, ensure_ascii=False, indent=2)
print(f"Training data: {len(TRAIN_DATA)} examples -> ./data/train_data.json")
print("For real training: use alpaca_zh_1k.json (1000 examples), ~30min on RTX 3070")
